# 10 (Capstone) - Logistic Attribution on the Real Campaign

Chapter 10 turns a table of binary outcomes and factor levels into a ranked, defensible statement about what causes failure. On a multi-component agent the decisive question is **which component to fix**, so the analysis is run **per component** -- the same deviance/Benjamini-Hochberg machinery, but on each tool's own correctness rather than on the end-to-end decision. Because the failures are few, a logistic fit is unstable and its odds ratios can diverge at zero-failure levels, so the robust report is the **empirical failure rate** at each factor level. The per-query decision is kept as a separate **overall** summary. This companion reads the pinned campaign; the diagnosis in context is notebook 12.6.

**Reference (run separately):** the calls that produced these numbers, over the rows of one balanced 120-run campaign. The seed case is excluded -- it was a blocking factor, not a property under test.

```python
from gmstest.evaluate import attribute, weak_link

# PER-COMPONENT: which input breaks which tool (empirical per-level failure rates).
# search_policy is graded by graph-truth (Sec. 12.5), so it is not label-attributed here.
for col in ('c_classify_complaint', 'c_flag_regulatory'):
    sub = [r for r in rows if r[col] is not None]
    attribute(sub, factors, metric=col)

# OVERALL per-query decision (right classification AND escalation).
overall = attribute(rows, factors)                  # metric defaults to 'correct'
blame   = weak_link(rows, workflow)
```

**What the next cell does** -- load the pinned campaign and define a table printer for the overall logistic attribution (one row per factor, with the worst level for a significant one).

In [ ]:
import json
from pathlib import Path

root = Path.cwd().parent / 'code'
while not (root/'data'/'capstone_run.json').exists() and root != root.parent:
    root = root.parent
D = json.loads((root/'data'/'capstone_run.json').read_text())
by_component = D['attribution_by_component']
overall = D['overall']
weak_link = D['weak_link']
WORKFLOW = ['classify_complaint', 'extract_facts', 'search_policy',
            'flag_regulatory', 'draft_response']
FACTOR_ORDER = ['reasoning_cue', 'clarity', 'entity_aliasing']

def show_factor_table(tbl):
    print(f"  {'factor':16s}{'G2':>8s}{'p_adj':>9s}{'pseudoR2':>10s}{'sig':>6s}   worst level (odds ratio)")
    for r in tbl:
        ors = r['odds_ratios']; worst = min(ors, key=ors.get)
        worst_str = f"{worst} ({ors[worst]})" if r['significant_adj'] else '---'
        print(f"  {r['factor']:16s}{r['G2']:>8.2f}{r['p_adj']:>9.4f}{r['pseudo_r2']:>10.3f}"
              f"{str(r['significant_adj']):>6s}   {worst_str}")
print('loaded per-component + overall attribution from', root/'data')

## Which input breaks which component

Each label-scored tool is analyzed on its own correctness, on the runs where it was graded. If a level concentrates a tool's failures it is the input condition to fix; if the rates are flat, no presentation factor drives the tool. Reported as empirical failure rates (failed / graded) -- robust to the separation that would blow up a logistic odds ratio on so few failures.

**What the next cell does** -- for each tool, print its graded/failure counts and, per factor (reasoning cue first), the `level=failed/n` failure rates.

In [ ]:
for tool in ('classify_complaint', 'flag_regulatory'):
    a = by_component[tool]
    print(f"{tool}  (graded {a['scored']}, failures {a['failures']})")
    for f in FACTOR_ORDER:
        levels = a['by_factor'][f]
        cells_str = '  '.join(f"{lvl}={v['failed']}/{v['n']}" for lvl, v in levels.items())
        print(f"    {f:16s}{cells_str}")
    print()

The classifier's nine failures spread across the levels -- the nearest gradient is `clarity` (ambiguous against clear), the reasoning cue is flat -- and no level concentrates them; the corrected fit certifies nothing (`clarity` nearest at $p_{adj}=0.96$). That is the actionable diagnosis a single end-to-end rate cannot give, here a null one: on this retrained agent no presentation factor drives a component. This reverses the earlier agent, where a downplaying cue dominated. The flagger fails once and has nothing to attribute. (`search_policy` is graded by graph-truth in Section 12.5, so it is not label-attributed here.)

## The overall decision, as a separate summary

The end-to-end view read as a whole is the per-query decision: the right classification and the right escalate/don't-escalate call. We report its rate, its 120-run factor attribution and the weak-link decomposition; the per-tool numbers are scored separately.

**What the next cell does** -- print the overall rate and definition, the overall logistic attribution table, and the weak-link blame per workflow tool.

In [ ]:
print(f"Overall decision accuracy: {overall['accuracy']:.3f}")
print(f"  {overall['definition']}\n")
print('Overall factor attribution (logistic, BH-corrected)')
show_factor_table(overall['attribution'])
print()
print('Weak-link blame (wrongly-decisioned runs -> first erring tool)')
for tool in WORKFLOW:
    if tool in weak_link:
        print(f"  {tool:20s} {weak_link[tool]}")
print(f"  {'total tool-attributed':20s} {sum(weak_link.values())}")

Run on the decision bit, the factor attribution finds **nothing** significant after correction, which here agrees with the per-component view rather than contradicting it: the retrained agent does not break on the presentation factors the design varies. The lesson stands on a multi-component agent: attribute the components separately, because a single bit can wash out a signal the per-component view would localize. Notebook 12.6 reads both as the capstone's diagnosis.

**What the next cell does** -- pin the SHAPE of the analysis, not volatile point estimates: each tool's per-level failure rates sum to its failure and graded counts; the overall decision is a valid rate with a well-formed logistic table; weak-link blame falls only on workflow tools.

In [ ]:
FACTORS = {'clarity', 'entity_aliasing', 'reasoning_cue'}
assert set(by_component) == {'classify_complaint', 'search_policy', 'flag_regulatory'}
for tool, a in by_component.items():
    assert set(a['by_factor']) == FACTORS
    for f, levels in a['by_factor'].items():
        assert sum(v['failed'] for v in levels.values()) == a['failures']
        assert sum(v['n'] for v in levels.values()) == a['scored']
FACT_KEYS = {'factor', 'G2', 'p_adj', 'pseudo_r2', 'significant_adj', 'odds_ratios'}
assert 0.0 <= overall['accuracy'] <= 1.0
assert {r['factor'] for r in overall['attribution']} == FACTORS
for r in overall['attribution']:
    assert FACT_KEYS <= set(r) and 0.0 <= r['p_adj'] <= 1.0
assert set(weak_link) <= set(WORKFLOW) and all(v > 0 for v in weak_link.values())
print('self-check passed')